# Gold Hospitality Companies Mart

**Purpose**: Top hospitality employers and company-level hiring patterns.

**Target Table**: `workspace.gold.gold_hospitality_companies`

**Grain**: One row per date per company within hospitality sector

**Sector Filter**: Filters to hospitality sector_sk values only

**Key Metrics**:
- Active jobs per company
- New postings and closures
- Hiring velocity
- Role diversity
- Geographic reach

**Data Sources**:
- `workspace.warehouse.fact_job_postings` (filtered to hospitality)
- `workspace.warehouse.dim_company`

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.gold.gold_hospitality_companies
USING DELTA
COMMENT 'Hospitality sector company-level hiring patterns'
AS

WITH hospitality_sector AS (
  SELECT sector_sk
  FROM workspace.warehouse.dim_sector
  WHERE sector_name IN ('Hospitality', 'Hotels & Resorts', 'Restaurants', 'Food & Beverage')
     OR sector_family = 'Hospitality'
),

base_metrics AS (
  SELECT
    f.posting_date_sk AS hospitality_date_sk,
    f.sector_sk,
    f.company_sk,
    
    -- MEASURES: Job counts
    COUNT(CASE WHEN f.active_flag = TRUE THEN 1 END) AS active_jobs,
    COUNT(CASE WHEN f.is_new_job = TRUE THEN 1 END) AS new_jobs,
    COUNT(CASE WHEN f.is_soft_delete = TRUE THEN 1 END) AS closed_jobs,
    COUNT(CASE WHEN f.is_new_job = TRUE THEN 1 END) - 
      COUNT(CASE WHEN f.is_soft_delete = TRUE THEN 1 END) AS net_change,
    
    -- MEASURES: Diversity metrics
    COUNT(DISTINCT CASE WHEN f.active_flag = TRUE THEN f.role_sk END) AS unique_roles,
    COUNT(DISTINCT CASE WHEN f.active_flag = TRUE THEN f.location_sk END) AS unique_locations,
    
    -- Total events
    COUNT(*) AS total_events
    
  FROM workspace.warehouse.fact_job_postings f
  INNER JOIN hospitality_sector hs ON f.sector_sk = hs.sector_sk
  WHERE f.posting_date_sk >= CAST(DATE_FORMAT(DATE_SUB(CURRENT_DATE(), 365), 'yyyyMMdd') AS INT)
  GROUP BY f.posting_date_sk, f.sector_sk, f.company_sk
),

temporal_enriched AS (
  SELECT
    bm.*,
    
    -- TEMPORAL METRICS: Prior period comparisons
    LAG(bm.active_jobs, 7) OVER (
      PARTITION BY bm.company_sk, bm.sector_sk
      ORDER BY bm.hospitality_date_sk
    ) AS prev_week_active_jobs,
    
    LAG(bm.active_jobs, 30) OVER (
      PARTITION BY bm.company_sk, bm.sector_sk
      ORDER BY bm.hospitality_date_sk
    ) AS prev_month_active_jobs,
    
    -- Week-to-date cumulative new jobs
    SUM(bm.new_jobs) OVER (
      PARTITION BY bm.company_sk, bm.sector_sk,
        YEAR(TO_DATE(CAST(bm.hospitality_date_sk AS STRING), 'yyyyMMdd')),
        WEEKOFYEAR(TO_DATE(CAST(bm.hospitality_date_sk AS STRING), 'yyyyMMdd'))
      ORDER BY bm.hospitality_date_sk
      ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
    ) AS wtd_new_jobs,
    
    -- Month-to-date cumulative new jobs
    SUM(bm.new_jobs) OVER (
      PARTITION BY bm.company_sk, bm.sector_sk,
        YEAR(TO_DATE(CAST(bm.hospitality_date_sk AS STRING), 'yyyyMMdd')),
        MONTH(TO_DATE(CAST(bm.hospitality_date_sk AS STRING), 'yyyyMMdd'))
      ORDER BY bm.hospitality_date_sk
      ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
    ) AS mtd_new_jobs,
    
    -- 7-day rolling average
    AVG(bm.new_jobs) OVER (
      PARTITION BY bm.company_sk, bm.sector_sk
      ORDER BY bm.hospitality_date_sk
      ROWS BETWEEN 6 PRECEDING AND CURRENT ROW
    ) AS rolling_7day_avg_new_jobs,
    
    -- 30-day rolling average
    AVG(bm.active_jobs) OVER (
      PARTITION BY bm.company_sk, bm.sector_sk
      ORDER BY bm.hospitality_date_sk
      ROWS BETWEEN 29 PRECEDING AND CURRENT ROW
    ) AS rolling_30day_avg_active_jobs,
    
    -- Rank companies by active jobs within each day
    DENSE_RANK() OVER (
      PARTITION BY bm.hospitality_date_sk, bm.sector_sk
      ORDER BY CASE WHEN bm.active_jobs > 0 THEN bm.active_jobs ELSE 0 END DESC
    ) AS daily_company_rank
    
  FROM base_metrics bm
)

SELECT
  -- DIMENSIONS
  te.hospitality_date_sk,
  te.sector_sk,
  te.company_sk,
  te.daily_company_rank,
  
  -- MEASURES
  te.active_jobs,
  te.new_jobs,
  te.closed_jobs,
  te.net_change,
  te.unique_roles,
  te.unique_locations,
  te.total_events,
  
  -- TEMPORAL METRICS
  te.wtd_new_jobs,
  te.mtd_new_jobs,
  CAST(te.rolling_7day_avg_new_jobs AS DECIMAL(10,2)) AS rolling_7day_avg_new_jobs,
  CAST(te.rolling_30day_avg_active_jobs AS DECIMAL(10,2)) AS rolling_30day_avg_active_jobs,
  
  -- DERIVED: Percent changes
  CASE 
    WHEN te.prev_week_active_jobs > 0 
    THEN CAST((te.active_jobs - te.prev_week_active_jobs) AS DECIMAL(10,4)) / te.prev_week_active_jobs
    ELSE NULL 
  END AS pct_change_vs_prev_week,
  
  CASE 
    WHEN te.prev_month_active_jobs > 0 
    THEN CAST((te.active_jobs - te.prev_month_active_jobs) AS DECIMAL(10,4)) / te.prev_month_active_jobs
    ELSE NULL 
  END AS pct_change_vs_prev_month,
  
  -- DERIVED: Hiring velocity
  CASE 
    WHEN te.active_jobs > 0 
    THEN CAST(te.new_jobs AS DECIMAL(10,4)) / te.active_jobs
    ELSE NULL 
  END AS hiring_velocity,
  
  -- DERIVED: Job churn rate
  CASE 
    WHEN te.prev_week_active_jobs > 0 
    THEN CAST(te.closed_jobs AS DECIMAL(10,4)) / te.prev_week_active_jobs
    ELSE NULL 
  END AS job_churn_rate,
  
  -- DERIVED: Role diversity score (roles per location)
  CASE 
    WHEN te.unique_locations > 0 
    THEN CAST(te.unique_roles AS DECIMAL(10,2)) / te.unique_locations
    ELSE te.unique_roles
  END AS role_diversity_score,
  
  -- DERIVED: Geographic reach indicator
  CASE 
    WHEN te.unique_locations >= 10 THEN 'National'
    WHEN te.unique_locations >= 5 THEN 'Regional'
    WHEN te.unique_locations >= 2 THEN 'Multi-City'
    ELSE 'Local'
  END AS geographic_reach,
  
  -- METADATA
  CURRENT_TIMESTAMP() AS created_at,
  UUID() AS batch_id
  
FROM temporal_enriched te
WHERE te.daily_company_rank <= 200  -- Keep top 200 companies per day for dashboard performance
ORDER BY te.hospitality_date_sk DESC, te.daily_company_rank;

In [0]:
%sql
-- Validation: Top hospitality employers
SELECT
  c.canonical_company_name,
  c.industry,
  SUM(ghc.active_jobs) AS total_active_jobs,
  SUM(ghc.new_jobs) AS total_new_jobs,
  ROUND(AVG(ghc.hiring_velocity), 4) AS avg_hiring_velocity,
  ROUND(AVG(ghc.job_churn_rate), 4) AS avg_churn_rate,
  MAX(ghc.unique_locations) AS max_locations,
  MAX(ghc.unique_roles) AS max_roles,
  COUNT(DISTINCT ghc.hospitality_date_sk) AS days_with_activity,
  ROUND(AVG(ghc.daily_company_rank), 1) AS avg_rank
FROM workspace.gold.gold_hospitality_companies ghc
JOIN workspace.warehouse.dim_company c ON ghc.company_sk = c.company_sk
GROUP BY c.canonical_company_name, c.industry
ORDER BY total_active_jobs DESC
LIMIT 30;